# KONTOGRAPH — 01: train the temporal GNN

Rungs 3 and 4 of the ablation ladder from `docs/evaluation-protocol.md`.

**Rung 3** is the TGN with its memory module *disabled* — temporal attention only.
**Rung 4** adds per-node memory. Running both is the point: it isolates exactly what
memory contributes, rather than assuming it helps.

Everything here is standalone. The `kg_*` packages are not installed on Kaggle, so the
split and the metrics are reimplemented — and `tests/unit/test_kaggle_parity.py` checks
them against the local implementations so the two cannot drift.

Before running: **phone-verify your Kaggle account**. It gates both GPU *and* notebook
internet, and without internet the install cell fails quietly and training dies several
cells later with an import error.

## Environment

In [ ]:
# PyG needs nothing but PyTorch since 2.3 -- torch-scatter, torch-sparse and
# pyg-lib are optional. So we install torch_geometric plainly and let it use
# whatever torch Kaggle ships, rather than pinning a CUDA wheel index.
#
# Pinning wheels against a CUDA version Kaggle has since moved past is the
# classic way these notebooks rot. Detecting instead of pinning is why this one
# should keep working as the base image changes.
import subprocess, sys, torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

# `cuda.is_available()` is NOT sufficient. Kaggle's P100 is compute capability
# sm_60, and current Kaggle PyTorch builds ship kernels for sm_70 and above --
# so CUDA reports available, every allocation succeeds, and the run dies at the
# first real kernel launch minutes later with an unhelpful error. Checking the
# capability against what this build actually compiled for turns that into an
# immediate, actionable failure.
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    supported = torch.cuda.get_arch_list()
    print(f"device: {name} (sm_{major}{minor})")
    print("this torch build supports:", " ".join(supported))
    if f"sm_{major}{minor}" not in supported:
        raise RuntimeError(
            f"{name} is sm_{major}{minor}, which this PyTorch build does not support "
            f"({' '.join(supported)}).\n"
            "Switch the accelerator: Notebook > Settings > Accelerator > GPU T4 x2 "
            "(T4 is sm_75). P100 is sm_60 and is too old for current Kaggle images.\n"
            "Failing now rather than at the first kernel launch several minutes in."
        )
else:
    raise RuntimeError(
        "No GPU. Notebook > Settings > Accelerator > GPU T4 x2. "
        "GPU and internet both require a phone-verified account."
    )

try:
    import torch_geometric
    print("torch_geometric", torch_geometric.__version__, "(already present)")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
    import torch_geometric
    print("torch_geometric", torch_geometric.__version__, "(installed)")

## Load the pushed dataset

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, json

# The dataset pushed by `kg kaggle push-data`. Located by CONTENT rather than by
# name: Kaggle does not guarantee where an attached dataset lands. This notebook
# previously globbed "/kaggle/input/*kontograph*" and fell back to
# "/kaggle/input" when that matched nothing -- and it did match nothing, because
# the dataset mounted under a nested "datasets/" directory. The fallback turned
# "the dataset is not attached" into a FileNotFoundError several frames deep in
# pandas, which is a much harder thing to read.
#
# So: search for the file that must exist, and if it does not, say so.
def find_dataset_root(marker="features.parquet"):
    base = Path("/kaggle/input")
    if not base.exists():
        raise RuntimeError("/kaggle/input does not exist -- is this running on Kaggle?")
    for candidate in sorted(base.rglob(marker)):
        return candidate.parent
    listing = [str(p.relative_to(base)) for p in sorted(base.rglob("*")) if p.is_file()][:40]
    raise FileNotFoundError(
        f"no {marker} anywhere under /kaggle/input.\n"
        f"The dataset is not attached, or is still processing.\n"
        f"Attach it via Notebook > Add Input, or re-run `kg kaggle push-kernel`.\n"
        f"What IS mounted: {listing or '(nothing)'}"
    )


ROOT = find_dataset_root()
print("input:", ROOT, "->", sorted(p.name for p in ROOT.iterdir()))

features = pd.read_parquet(ROOT / "features.parquet")
labels = pd.read_parquet(ROOT / "labels.parquet")
manifest = json.loads((ROOT / "manifest.json").read_text())

print(f"{len(features):,} events x {features.shape[1]} columns")
print(f"{len(labels):,} labels")
print("dataset fingerprint:", manifest["config_fingerprint"])
print("positive rate (oracle):", manifest["rates"]["positive_rate"])
print("criminal label coverage:", manifest["rates"]["criminal_label_coverage"])

## The chronological split, with label availability

Three separate mechanisms, routinely conflated:

- **chronological split** — never random; a random split puts an account's later
  transactions in train and its earlier ones in test
- **purge / embargo** — one day either side of each boundary, addressing *correlated
  errors*, not leakage (the feature layer already prevents that)
- **label availability** — a label counts only if it was **decided** before the boundary

In [ ]:
# The protocol's split, reimplemented here rather than imported: this notebook
# must run standalone on Kaggle, where the kg_* packages are not installed.
# It is checked against the local implementation by
# tests/unit/test_kaggle_parity.py, so the two cannot drift silently.
DAY = 86_400_000
BAND = DAY                     # purge/embargo, protocol section 2
FRACTIONS = (0.60, 0.20, 0.20)

ts = features["acceptance_ts"].to_numpy()
start, end = int(ts.min()), int(ts.max()) + 1
span = end - start
t1 = start + int(span * FRACTIONS[0])
t2 = start + int(span * (FRACTIONS[0] + FRACTIONS[1]))

fold = np.full(len(ts), "test", dtype=object)
fold[ts < t1] = "train"
fold[(ts >= t1) & (ts < t2)] = "validation"
fold[(ts >= t1 - BAND) & (ts < t1 + BAND)] = "dropped"
fold[(ts >= t2 - BAND) & (ts < t2 + BAND)] = "dropped"

# LABEL AVAILABILITY -- the rule that actually stops time travel.
# A label counts for a fold only if it was DECIDED before that fold's boundary.
# Filtering on the transaction's own timestamp is the common mistake and it
# hands the model answers nobody had yet.
POSITIVE = {"CONFIRMED_FRAUD", "CONFIRMED_LAUNDERING"}

def targets(cutoff):
    available = labels if cutoff is None else labels[labels["decided_ts"] < cutoff]
    positive = set(available.loc[available["label"].isin(POSITIVE), "subject_id"])
    return features["tx_id"].isin(positive).to_numpy().astype(np.float32)

y = {
    "train": targets(t1),
    "validation": targets(t2),
    "test": targets(None),   # scored after the fact
}
for name in ("train", "validation", "test"):
    mask = fold == name
    print(f"{name:11} {mask.sum():>8,} rows  {int(y[name][mask].sum()):>5} positive")
print(f"dropped     {(fold == 'dropped').sum():>8,} (purge + embargo)")

## Build the temporal graph

In [ ]:
FEATURE_COLUMNS = [
    c for c in features.columns
    if c not in {"tx_id", "acceptance_ts", "debtor_account_id", "creditor_account_id",
                 "amount_cents", "event_date", "source_dataset"}
]
print(f"{len(FEATURE_COLUMNS)} model inputs")

order = np.lexsort((features["tx_id"].to_numpy(), ts))
accounts = sorted(set(features["debtor_account_id"]) | set(features["creditor_account_id"]))
index = {a: i for i, a in enumerate(accounts)}

X = features[FEATURE_COLUMNS].to_numpy(dtype=np.float32)[order]
# Standardised: the time encoder and attention are scale-sensitive, and the raw
# features span cents to milliseconds.
X = (X - X.mean(0)) / (X.std(0) + 1e-6)

src = np.array([index[a] for a in features["debtor_account_id"].to_numpy()[order]], dtype=np.int64)
dst = np.array([index[a] for a in features["creditor_account_id"].to_numpy()[order]], dtype=np.int64)
# Integer SECONDS: PyG's TGNMemory keeps last_update as a Long buffer, and
# millisecond magnitudes push the time encoder into a bad numeric regime.
t_sec = (ts[order] // 1000).astype(np.int64)
fold_ordered = fold[order]
y_ordered = {k: v[order] for k, v in y.items()}
tx_ordered = features["tx_id"].to_numpy()[order]

print(f"graph: {len(accounts):,} nodes, {len(src):,} edges, edge_dim={X.shape[1]}")

## The model

In [ ]:
import torch
from torch.nn import Linear, ReLU, Dropout, Sequential
from torch_geometric.nn import TransformerConv
from torch_geometric.nn.models.tgn import (
    TGNMemory, IdentityMessage, LastAggregator, LastNeighborLoader, TimeEncoder,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEMORY_DIM = EMBEDDING_DIM = TIME_DIM = 100
NEIGHBOURS, HEADS, DROPOUT = 10, 2, 0.1
torch.manual_seed(20251009)

class GraphAttentionEmbedding(torch.nn.Module):
    """Temporal attention over a node's most recent neighbours.

    The time delta on each edge is encoded and concatenated to the edge
    features, so attention can tell "paid me a minute ago" from "a month ago".
    Without it this is a static GNN over a recency-truncated neighbourhood.
    """
    def __init__(self, in_channels, out_channels, msg_dim, time_enc):
        super().__init__()
        self.time_enc = time_enc
        self.conv = TransformerConv(in_channels, out_channels // HEADS, heads=HEADS,
                                    dropout=DROPOUT,
                                    edge_dim=msg_dim + time_enc.out_channels)
    def forward(self, x, last_update, edge_index, t, msg):
        rel_t = self.time_enc((last_update[edge_index[0]] - t).to(x.dtype))
        return self.conv(x, edge_index, torch.cat([rel_t, msg], dim=-1))

class EdgeDecoder(torch.nn.Module):
    """Binary classifier over both endpoint embeddings plus the edge features.

    The transaction's own features are included deliberately: the graph supplies
    context, but the payment still carries the amount, channel and VoP outcome,
    and dropping them to look more graph-native would throw away real signal.
    """
    def __init__(self, emb, edge_dim):
        super().__init__()
        self.net = Sequential(Linear(emb * 2 + edge_dim, emb), ReLU(), Dropout(DROPOUT),
                              Linear(emb, emb // 2), ReLU(), Linear(emb // 2, 1))
    def forward(self, z_src, z_dst, edge_feat):
        return self.net(torch.cat([z_src, z_dst, edge_feat], dim=-1)).squeeze(-1)

def build(use_memory=True):
    edge_dim = X.shape[1]
    memory = TGNMemory(
        len(accounts), edge_dim, MEMORY_DIM, TIME_DIM,
        message_module=IdentityMessage(edge_dim, MEMORY_DIM, TIME_DIM),
        aggregator_module=LastAggregator(),
    ).to(DEVICE) if use_memory else None
    time_enc = memory.time_enc if memory is not None else TimeEncoder(TIME_DIM).to(DEVICE)
    emb = GraphAttentionEmbedding(
        MEMORY_DIM if use_memory else edge_dim, EMBEDDING_DIM, edge_dim, time_enc
    ).to(DEVICE)
    dec = EdgeDecoder(EMBEDDING_DIM, edge_dim).to(DEVICE)
    loader = LastNeighborLoader(len(accounts), size=NEIGHBOURS, device=DEVICE)
    return memory, emb, dec, loader

print("device:", DEVICE)

## Training loop

In [ ]:
SRC = torch.as_tensor(src, device=DEVICE)
DST = torch.as_tensor(dst, device=DEVICE)
TS  = torch.as_tensor(t_sec, device=DEVICE)          # Long, matches last_update
FEAT = torch.as_tensor(X, dtype=torch.float, device=DEVICE)

BATCH, EPOCHS, LR, POS_WEIGHT = 200, 5, 1e-4, 50.0

def batches(rows, size=BATCH):
    for i in range(0, len(rows), size):
        yield rows[i:i + size]

def run_training(use_memory=True, epochs=EPOCHS, tag="tgn"):
    memory, emb, dec, loader = build(use_memory)
    modules = [m for m in (memory, emb, dec) if m is not None]
    params = [p for m in modules for p in m.parameters()]
    opt = torch.optim.Adam(params, lr=LR)
    crit = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEVICE))

    train_rows = np.flatnonzero(fold_ordered == "train")
    y_train = torch.as_tensor(y_ordered["train"], device=DEVICE)
    history = []

    for epoch in range(epochs):
        for m in modules: m.train()
        # Reset every epoch: the model must relearn the stream from its start,
        # exactly as it will at deployment.
        if memory is not None: memory.reset_state()
        loader.reset_state()
        total, seen = 0.0, 0

        for rows in batches(train_rows):
            opt.zero_grad()
            bs, bd = SRC[rows], DST[rows]
            nodes = torch.cat([bs, bd]).unique()
            n_id, edge_index, e_id = loader(nodes)
            assoc = torch.empty(len(accounts), dtype=torch.long, device=DEVICE)
            assoc[n_id] = torch.arange(n_id.size(0), device=DEVICE)

            if memory is not None:
                z, last_update = memory(n_id)
            else:
                z = torch.zeros(n_id.size(0), X.shape[1], device=DEVICE)
                last_update = torch.zeros(n_id.size(0), dtype=torch.long, device=DEVICE)

            emb_out = emb(z, last_update, edge_index,
                          TS[rows][0].expand(edge_index.size(1)),
                          FEAT[e_id] if e_id.numel() else FEAT[:0])
            logits = dec(emb_out[assoc[bs]], emb_out[assoc[bd]], FEAT[rows])
            loss = crit(logits, y_train[rows])
            loss.backward(); opt.step()

            # State updated AFTER the step: this batch never contributed to the
            # embedding that predicted it.
            if memory is not None:
                memory.update_state(bs, bd, TS[rows], FEAT[rows]); memory.detach()
            loader.insert(bs, bd)
            total += float(loss) * len(rows); seen += len(rows)

        history.append({"epoch": epoch, "train_loss": total / max(1, seen)})
        print(f"  {tag} epoch {epoch}  loss {total / max(1, seen):.5f}")
    return (memory, emb, dec, loader), history

## Scoring and metrics

In [ ]:
@torch.no_grad()
def score(bundle, rows, replay_from=None):
    """Score rows, replaying earlier events first to build the memory they need.

    Resetting and replaying is not optional. Scoring without a reset means the
    test fold is scored against a memory that already absorbed it -- a
    spectacular, meaningless number with no trace in the metrics. Scoring
    without the replay means a cold memory, which understates the model.
    """
    memory, emb, dec, loader = bundle
    for m in (memory, emb, dec):
        if m is not None: m.eval()
    if memory is not None: memory.reset_state()
    loader.reset_state()

    if replay_from is not None:
        for warm in batches(replay_from):
            if memory is not None:
                memory.update_state(SRC[warm], DST[warm], TS[warm], FEAT[warm])
            loader.insert(SRC[warm], DST[warm])

    out = []
    for rows_b in batches(rows):
        bs, bd = SRC[rows_b], DST[rows_b]
        nodes = torch.cat([bs, bd]).unique()
        n_id, edge_index, e_id = loader(nodes)
        assoc = torch.empty(len(accounts), dtype=torch.long, device=DEVICE)
        assoc[n_id] = torch.arange(n_id.size(0), device=DEVICE)
        if memory is not None:
            z, last_update = memory(n_id)
        else:
            z = torch.zeros(n_id.size(0), X.shape[1], device=DEVICE)
            last_update = torch.zeros(n_id.size(0), dtype=torch.long, device=DEVICE)
        emb_out = emb(z, last_update, edge_index,
                      TS[rows_b][0].expand(edge_index.size(1)),
                      FEAT[e_id] if e_id.numel() else FEAT[:0])
        out.append(dec(emb_out[assoc[bs]], emb_out[assoc[bd]], FEAT[rows_b]).cpu().numpy())
        if memory is not None:
            memory.update_state(bs, bd, TS[rows_b], FEAT[rows_b])
        loader.insert(bs, bd)
    return 1 / (1 + np.exp(-np.clip(np.concatenate(out), -50, 50)))

def average_precision(y_true, scores):
    """PR-AUC with explicit tie handling -- the protocol's headline metric."""
    y_true = np.asarray(y_true, dtype=np.int8); scores = np.asarray(scores, dtype=float)
    pos = int(y_true.sum())
    if pos == 0: return 0.0
    o = np.argsort(-scores, kind="stable")
    st, ss = y_true[o], scores[o]
    tp = np.cumsum(st); ranks = np.arange(1, len(st) + 1)
    precision, recall = tp / ranks, tp / pos
    last = np.ones(len(st), bool); last[:-1] = ss[:-1] != ss[1:]
    return float(np.sum(precision[last] * np.diff(np.concatenate([[0.0], recall[last]]))))

def precision_at_k(y_true, scores, k):
    k = min(k, len(y_true))
    return float(np.asarray(y_true)[np.argsort(-scores, kind="stable")[:k]].mean())

## Run both rungs and write the artifact bundle

In [ ]:
import time, json

train_rows = np.flatnonzero(fold_ordered == "train")
val_rows   = np.flatnonzero(fold_ordered == "validation")
test_rows  = np.flatnonzero(fold_ordered == "test")
capacity_k = max(1, int(200 * (t_sec[test_rows].max() - t_sec[test_rows].min()) / 86400))

results = {}
for use_memory, tag in [(True, "tgn_memory"), (False, "tgn_no_memory")]:
    started = time.time()
    bundle, history = run_training(use_memory=use_memory, tag=tag)
    scores = score(bundle, test_rows, replay_from=np.concatenate([train_rows, val_rows]))
    yt = y_ordered["test"][test_rows]
    results[tag] = {
        "pr_auc": average_precision(yt, scores),
        f"precision_at_200_per_day": precision_at_k(yt, scores, capacity_k),
        "use_memory": use_memory,
        "train_seconds": round(time.time() - started, 1),
        "history": history,
    }
    print(f"{tag}: PR-AUC {results[tag]['pr_auc']:.4f}  "
          f"P@k {results[tag]['precision_at_200_per_day']:.4f}  "
          f"({results[tag]['train_seconds']:.0f}s)")
    np.save(f"/kaggle/working/scores_{tag}.npy", scores)
    if use_memory:
        memory, emb, dec, _ = bundle
        state = {"embedding": emb.state_dict(), "decoder": dec.state_dict()}
        if memory is not None: state["memory"] = memory.state_dict()
        torch.save(state, "/kaggle/working/tgn.pt")

# The Kaggle -> local contract: a fixed-schema metrics.json plus checkpoints.
# `kg kaggle pull-artifacts` reads exactly this.
payload = {
    "dataset_fingerprint": manifest["config_fingerprint"],
    "feature_columns": FEATURE_COLUMNS,
    "capacity_k": capacity_k,
    "test_positives": int(y_ordered["test"][test_rows].sum()),
    "test_rows": int(len(test_rows)),
    "results": results,
}
Path("/kaggle/working/metrics.json").write_text(json.dumps(payload, indent=2, default=float))
np.save("/kaggle/working/test_tx_ids.npy", tx_ordered[test_rows])
print("\nwrote /kaggle/working/metrics.json, tgn.pt, scores_*.npy")